In [5]:
import numpy as np
import pandas as pd
from itertools import product as iproduct

In [ ]:
rng = np.random.default_rng(42)

BUDGET = 1_000_000
N_TRIALS = 100_000

PRODUCTS = {
    "Thermalite":     {"min_r": -0.10, "max_r": 0.60, "anchor": 0.15,
                       "rational": 0.9,  "normal": 0.7,  "idiot_type": "random"},
    "LavaCakes":      {"min_r": -0.60, "max_r": 0.10, "anchor": -0.15,
                       "rational": -0.9, "normal": -0.8, "idiot_type": "random"},
    "Pyroflex":       {"min_r": -0.50, "max_r": 0.05, "anchor": -0.10,
                       "rational": -0.8, "normal": -0.6, "idiot_type": "random"},
    "Sulfur":         {"min_r": -0.05, "max_r": 0.30, "anchor": 0.10,
                       "rational": 0.7,  "normal": 0.3,  "idiot_type": "random"},
    "LavaFountainPen":{"min_r": -0.10, "max_r": 0.40, "anchor": 0.05,
                       "rational": 0.4,  "normal": 0.6,  "idiot_type": "hype"},
    "VolcanicIncense":{"min_r": -0.30, "max_r": 0.40, "anchor": 0.00,
                       "rational": -0.5, "normal": 0.3,  "idiot_type": "nostralico"},
    "AshesOfPhoenix": {"min_r": -0.30, "max_r": 0.15, "anchor": -0.05,
                       "rational": -0.5, "normal": -0.2, "idiot_type": "random"},
    "ObsidianCutlery":{"min_r": -0.25, "max_r": 0.10, "anchor": -0.05,
                       "rational": -0.5, "normal": 0.0,  "idiot_type": "random"},
    "ScoriaPaste":    {"min_r": -0.10, "max_r": 0.25, "anchor": 0.05,
                       "rational": 0.3,  "normal": 0.0,  "idiot_type": "random"},
}

# ── player segment sizes ──────────────────────────────────────────
SEGMENTS = {
    "rational": 2000,
    "normal":   1000,
    "idiot":     500,
}

ALLOC_VALUES = np.arange(0, 35, 5)  

def idiot_vote(idiot_type, n):
    if idiot_type == "hype":
        return rng.uniform(0.5, 1.0, n)
    elif idiot_type == "nostralico":
        return rng.uniform(0.6, 1.0, n)
    else:
        return rng.uniform(-1.0, 1.0, n)

def simulate_crowd_pressure(prod_cfg, n_rational, n_normal, n_idiot, n_trials):


    r_votes = rng.normal(prod_cfg["rational"], 0.15, (n_trials, n_rational))
    r_votes = np.clip(r_votes, -1, 1)

    n_votes = rng.normal(prod_cfg["normal"], 0.30, (n_trials, n_normal))
    n_votes = np.clip(n_votes, -1, 1)

    if prod_cfg["idiot_type"] == "hype":
        i_votes = rng.uniform(0.5, 1.0, (n_trials, n_idiot))
    elif prod_cfg["idiot_type"] == "nostralico":
        i_votes = rng.uniform(0.6, 1.0, (n_trials, n_idiot))
    else:
        i_votes = rng.uniform(-1.0, 1.0, (n_trials, n_idiot))

    total_votes = n_rational + n_normal + n_idiot
    pressure = (r_votes.sum(axis=1) + n_votes.sum(axis=1) + i_votes.sum(axis=1)) / total_votes

    return np.clip(pressure, -1, 1)

def pressure_to_return(pressure, prod_cfg):

    anchor = prod_cfg["anchor"]
    min_r = prod_cfg["min_r"]
    max_r = prod_cfg["max_r"]

    return np.where(
        pressure >= 0,
        anchor + pressure * (max_r - anchor),
        anchor + pressure * (anchor - min_r)
    )

def fee(vol_pct):
    return (vol_pct / 100) ** 2 * BUDGET

def run_simulation(n_rational=2000, n_normal=1000, n_idiot=500, n_trials=N_TRIALS):

    product_returns = {}
    for name, cfg in PRODUCTS.items():
        pressure = simulate_crowd_pressure(cfg, n_rational, n_normal, n_idiot, n_trials)
        product_returns[name] = pressure_to_return(pressure, cfg)

    rows = []
    for name in PRODUCTS:
        returns = product_returns[name]
        for alloc in ALLOC_VALUES:
            if alloc == 0:
                pnl = np.zeros(n_trials)
            else:
                mean_return = np.mean(returns)
                direction = 1 if mean_return >= 0 else -1
                pnl = direction * alloc/100 * BUDGET * returns - fee(alloc)

            rows.append({
                "product": name,
                "alloc_pct": alloc,
                "mean_pnl": round(np.mean(pnl), 2),
                "std_pnl": round(np.std(pnl), 2),
                "p10": round(np.percentile(pnl, 10), 2),
                "p90": round(np.percentile(pnl, 90), 2),
                "sharpe": round(np.mean(pnl) / np.std(pnl), 4) if np.std(pnl) > 0 else 0,
            })

    return pd.DataFrame(rows)

SEGMENT_SCENARIOS = [
    (2000, 1000, 500),   # base case
    (1500, 1000, 1000),  # more idiots
    (2500, 700,  300),   # more rational
    (1800, 1200, 500),   # more normal
]

all_rows = []
for n_rat, n_norm, n_id in SEGMENT_SCENARIOS:
    df = run_simulation(n_rat, n_norm, n_id)
    df["scenario"] = f"rat{n_rat}_norm{n_norm}_id{n_id}"
    all_rows.append(df)

df_all = pd.concat(all_rows, ignore_index=True)

best = df_all.loc[df_all.groupby(["scenario", "product"])["mean_pnl"].idxmax()]
print(best[["scenario", "product", "alloc_pct", "mean_pnl", "std_pnl", "sharpe"]].to_string(index=False))

avg_best = best.groupby("product")["alloc_pct"].mean().sort_values(ascending=False)
print(avg_best)

base = df_all[df_all["scenario"] == "rat2000_norm1000_id500"]
best_base = base.loc[base.groupby("product")["mean_pnl"].idxmax()]
print(best_base[["product", "alloc_pct", "mean_pnl", "p10", "p90", "sharpe"]].to_string(index=False))
total_alloc = best_base["alloc_pct"].sum()
print(f"\nTotal allocation: {total_alloc}%")
print(f"Total fees: {sum(fee(a) for a in best_base['alloc_pct']):,.0f}")


── optimal allocation per product (by mean PnL) ─────────────────
               scenario         product  alloc_pct  mean_pnl  std_pnl   sharpe
rat1500_norm1000_id1000  AshesOfPhoenix          5   3391.35    76.17  44.5231
rat1500_norm1000_id1000       LavaCakes         20  43248.03   520.06  83.1589
rat1500_norm1000_id1000 LavaFountainPen         10  14373.11   114.40 125.6338
rat1500_norm1000_id1000 ObsidianCutlery          5   2143.00    61.18  35.0269
rat1500_norm1000_id1000        Pyroflex         15  22977.58   359.90  63.8439
rat1500_norm1000_id1000     ScoriaPaste          5   1285.98    60.87  21.1280
rat1500_norm1000_id1000          Sulfur         10   7697.36   121.95  63.1172
rat1500_norm1000_id1000      Thermalite         20  41197.72   528.85  77.9001
rat1500_norm1000_id1000 VolcanicIncense          0      0.00     0.00   0.0000
 rat1800_norm1200_id500  AshesOfPhoenix          5   4069.81    63.33  64.2646
 rat1800_norm1200_id500       LavaCakes         25  54869.22   5

In [ ]:
import numpy as np
import pandas as pd
from itertools import product as iproduct

rng = np.random.default_rng(42)

BUDGET = 1_000_000
N_TRIALS = 50_000  

PRODUCTS = {
    "Thermalite":      {"min_r": -0.10, "max_r": 0.60, "anchor": 0.15,
                        "rational": 0.9,  "normal": 0.7,  "idiot_type": "random",
                        "awareness": {"rational": 0.90, "normal": 0.70, "idiot": 0.20}},
    "LavaCakes":       {"min_r": -0.60, "max_r": 0.10, "anchor": -0.15,
                        "rational": -0.9, "normal": -0.8, "idiot_type": "random",
                        "awareness": {"rational": 0.90, "normal": 0.75, "idiot": 0.15}},
    "Pyroflex":        {"min_r": -0.50, "max_r": 0.05, "anchor": -0.10,
                        "rational": -0.8, "normal": -0.6, "idiot_type": "random",
                        "awareness": {"rational": 0.60, "normal": 0.30, "idiot": 0.05}},
    "Sulfur":          {"min_r": -0.05, "max_r": 0.30, "anchor": 0.10,
                        "rational": 0.7,  "normal": 0.3,  "idiot_type": "random",
                        "awareness": {"rational": 0.50, "normal": 0.20, "idiot": 0.05}},
    "LavaFountainPen": {"min_r": -0.10, "max_r": 0.40, "anchor": 0.05,
                        "rational": 0.4,  "normal": 0.6,  "idiot_type": "hype",
                        "awareness": {"rational": 0.60, "normal": 0.70, "idiot": 0.80}},
    "VolcanicIncense": {"min_r": -0.30, "max_r": 0.40, "anchor": 0.00,
                        "rational": -0.5, "normal": 0.3,  "idiot_type": "nostralico",
                        "awareness": {"rational": 0.30, "normal": 0.15, "idiot": 0.85}},
    "AshesOfPhoenix":  {"min_r": -0.30, "max_r": 0.15, "anchor": -0.05,
                        "rational": -0.5, "normal": -0.2, "idiot_type": "random",
                        "awareness": {"rational": 0.50, "normal": 0.40, "idiot": 0.20}},
    "ObsidianCutlery": {"min_r": -0.25, "max_r": 0.10, "anchor": -0.05,
                        "rational": -0.5, "normal": 0.0,  "idiot_type": "random",
                        "awareness": {"rational": 0.30, "normal": 0.10, "idiot": 0.05}},
    "ScoriaPaste":     {"min_r": -0.10, "max_r": 0.25, "anchor": 0.05,
                        "rational": 0.3,  "normal": 0.0,  "idiot_type": "random",
                        "awareness": {"rational": 0.20, "normal": 0.10, "idiot": 0.05}},
}

SEGMENT_SCENARIOS = [
    (2000, 1000, 500),
    (1500, 1000, 1000),
    (2500, 700,  300),
    (1800, 1200, 500),
    (1200, 1300, 1000),  # pessimistic — more idiots and normal
]

def fee(vol_pct):
    return (vol_pct / 100) ** 2 * BUDGET

def simulate_crowd_with_awareness(prod_cfg, n_rational, n_normal, n_idiot, n_trials):
    awareness = prod_cfg["awareness"]

    # rational
    aware_r = rng.random((n_trials, n_rational)) < awareness["rational"]
    r_signal = rng.normal(prod_cfg["rational"], 0.15, (n_trials, n_rational))
    r_random = rng.uniform(-1, 1, (n_trials, n_rational))
    r_votes = np.where(aware_r, r_signal, r_random)
    r_votes = np.clip(r_votes, -1, 1)

    # normal
    aware_n = rng.random((n_trials, n_normal)) < awareness["normal"]
    n_signal = rng.normal(prod_cfg["normal"], 0.30, (n_trials, n_normal))
    n_random = rng.uniform(-1, 1, (n_trials, n_normal))
    n_votes = np.where(aware_n, n_signal, n_random)
    n_votes = np.clip(n_votes, -1, 1)

    # idiots
    aware_i = rng.random((n_trials, n_idiot)) < awareness["idiot"]
    if prod_cfg["idiot_type"] == "hype":
        idiot_signal = rng.uniform(0.5, 1.0, (n_trials, n_idiot))
    elif prod_cfg["idiot_type"] == "nostralico":
        idiot_signal = rng.uniform(0.6, 1.0, (n_trials, n_idiot))
    else:
        idiot_signal = rng.uniform(-1, 1, (n_trials, n_idiot))
    i_random = rng.uniform(-1, 1, (n_trials, n_idiot))
    i_votes = np.where(aware_i, idiot_signal, i_random)
    i_votes = np.clip(i_votes, -1, 1)

    total = n_rational + n_normal + n_idiot
    pressure = (r_votes.sum(axis=1) + n_votes.sum(axis=1) + i_votes.sum(axis=1)) / total
    return np.clip(pressure, -1, 1)

def pressure_to_return(pressure, prod_cfg):
    anchor = prod_cfg["anchor"]
    min_r  = prod_cfg["min_r"]
    max_r  = prod_cfg["max_r"]
    return np.where(
        pressure >= 0,
        anchor + pressure * (max_r - anchor),
        anchor + pressure * (anchor - min_r)
    )

def simulate_returns(n_rational, n_normal, n_idiot, n_trials=N_TRIALS):
    product_returns = {}
    product_directions = {}
    for name, cfg in PRODUCTS.items():
        pressure = simulate_crowd_with_awareness(cfg, n_rational, n_normal, n_idiot, n_trials)
        returns = pressure_to_return(pressure, cfg)
        product_returns[name] = returns
        product_directions[name] = 1 if np.mean(returns) >= 0 else -1
    return product_returns, product_directions

def portfolio_pnl(alloc_dict, product_returns, product_directions):
    pnl = np.zeros(N_TRIALS)
    for name, alloc in alloc_dict.items():
        if alloc == 0:
            continue
        returns = product_returns[name]
        direction = product_directions[name]
        pnl += direction * (alloc / 100) * BUDGET * returns - fee(alloc)
    return pnl


HIGH_CONVICTION  = ["Thermalite", "LavaCakes", "Pyroflex", "Sulfur"]
LOW_CONVICTION   = ["LavaFountainPen", "VolcanicIncense", "AshesOfPhoenix", 
                    "ObsidianCutlery", "ScoriaPaste"]

ALLOC_COARSE = list(range(0, 31, 5))

def efficient_portfolio_sweep(product_returns, product_directions, max_alloc=100):
    products = list(PRODUCTS.keys())
    n = len(products)
    
    best_mean = -np.inf
    best_portfolio = None
    best_pnl = None

    print("Stage 1: coarse sweep high conviction...")
    stage1_best = {p: 0 for p in products}
    stage1_best_mean = -np.inf
    
    for allocs in iproduct(ALLOC_COARSE, repeat=len(HIGH_CONVICTION)):
        if sum(allocs) > max_alloc:
            continue
        alloc_dict = {p: 0 for p in products}
        for p, a in zip(HIGH_CONVICTION, allocs):
            alloc_dict[p] = a
        pnl = portfolio_pnl(alloc_dict, product_returns, product_directions)
        m = np.mean(pnl)
        if m > stage1_best_mean:
            stage1_best_mean = m
            stage1_best = alloc_dict.copy()
    
    print(f"Stage 1 best mean: {stage1_best_mean:,.0f}")
    print(f"Stage 1 allocations: { {k:v for k,v in stage1_best.items() if v>0} }")

    print("\nStage 2: sweep low conviction with remaining budget...")
    remaining = max_alloc - sum(stage1_best[p] for p in HIGH_CONVICTION)
    low_alloc_range = list(range(0, min(remaining + 1, 21), 5))
    
    for allocs in iproduct(low_alloc_range, repeat=len(LOW_CONVICTION)):
        if sum(allocs) > remaining:
            continue
        alloc_dict = stage1_best.copy()
        for p, a in zip(LOW_CONVICTION, allocs):
            alloc_dict[p] = a
        pnl = portfolio_pnl(alloc_dict, product_returns, product_directions)
        m = np.mean(pnl)
        if m > best_mean:
            best_mean = m
            best_portfolio = alloc_dict.copy()
            best_pnl = pnl.copy()
    
    return best_portfolio, best_mean, best_pnl

def fine_tune(best_portfolio, product_returns, product_directions, window=4):
    print("\nStage 3: fine tuning at 1% steps...")
    best_mean = np.mean(portfolio_pnl(best_portfolio, product_returns, product_directions))
    improved = True
    
    while improved:
        improved = False
        for name in PRODUCTS:
            current = best_portfolio[name]
            for delta in range(-window, window + 1):
                if delta == 0:
                    continue
                new_alloc = current + delta
                if new_alloc < 0:
                    continue
                test = best_portfolio.copy()
                test[name] = new_alloc
                if sum(test.values()) > 100:
                    continue
                pnl = portfolio_pnl(test, product_returns, product_directions)
                m = np.mean(pnl)
                if m > best_mean:
                    best_mean = m
                    best_portfolio = test.copy()
                    improved = True
    
    return best_portfolio, best_mean

def fine_tune_variable_budget(best_portfolio, product_returns, product_directions):
    print("\nStage 3: Fine-tuning (Finding Optimal Total Budget)...")
    current_portfolio = best_portfolio.copy()
    for name in PRODUCTS:
        if name not in current_portfolio:
            current_portfolio[name] = 0
            
    best_mean = np.mean(portfolio_pnl(current_portfolio, product_returns, product_directions))
    
    improved = True
    while improved:
        improved = False
        for name in PRODUCTS:
            for delta in [-3, -2, -1, 1, 2, 3]:
                test_portfolio = current_portfolio.copy()
                test_portfolio[name] += delta

                if test_portfolio[name] < 0 or sum(test_portfolio.values()) > 100:
                    continue
                
                new_mean = np.mean(portfolio_pnl(test_portfolio, product_returns, product_directions))
                
                if new_mean > best_mean:
                    best_mean = new_mean
                    current_portfolio = test_portfolio.copy()
                    print(f"  Nudge: {name} by {delta:+}% | Total Alloc: {sum(current_portfolio.values())}% | Mean: {best_mean:,.0f}")
                    improved = True
                    break
            if improved: break
            
    return current_portfolio, best_mean

all_results = []

for n_rat, n_norm, n_id in SEGMENT_SCENARIOS:
    scenario_name = f"rat{n_rat}_norm{n_norm}_id{n_id}"
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    print(f"{'='*60}")
    
    product_returns, product_directions = simulate_returns(n_rat, n_norm, n_id)
    
    print(f"\nExpected returns per product:")
    for name, returns in product_returns.items():
        print(f"  {name:<20} mean return: {np.mean(returns):>7.3f}  direction: {'BUY' if product_directions[name]==1 else 'SELL'}")
    
    best_portfolio, best_mean, best_pnl = efficient_portfolio_sweep(
        product_returns, product_directions
    )
    
    best_portfolio, best_mean = fine_tune(
        best_portfolio, product_returns, product_directions
    )
    
    best_portfolio, best_mean =fine_tune_variable_budget(
        best_portfolio, product_returns, product_directions
    )
    
    final_pnl = portfolio_pnl(best_portfolio, product_returns, product_directions)
    
    print(f"\n── best portfolio ───────────────────────────────────────")
    for name, alloc in best_portfolio.items():
        if alloc > 0:
            direction = "BUY" if product_directions[name] == 1 else "SELL"
            f = fee(alloc)
            print(f"  {name:<20} {direction:<5} {alloc:>3}%  fee: {f:>8,.0f}")
    
    total_alloc = sum(best_portfolio.values())
    total_fees  = sum(fee(a) for a in best_portfolio.values())
    
    print(f"\n  Total allocation: {total_alloc}%")
    print(f"  Total fees:       {total_fees:>10,.0f}")
    print(f"  Mean PnL:         {np.mean(final_pnl):>10,.0f}")
    print(f"  Std PnL:          {np.std(final_pnl):>10,.0f}")
    print(f"  P10:              {np.percentile(final_pnl, 10):>10,.0f}")
    print(f"  P90:              {np.percentile(final_pnl, 90):>10,.0f}")
    print(f"  Sharpe:           {np.mean(final_pnl)/np.std(final_pnl):>10.4f}")
    
    all_results.append({
        "scenario": scenario_name,
        "portfolio": best_portfolio,
        "mean_pnl": round(np.mean(final_pnl), 0),
        "std_pnl": round(np.std(final_pnl), 0),
        "total_alloc": total_alloc,
        "total_fees": round(total_fees, 0),
    })

print(f"\n{'='*60}")
print("CROSS SCENARIO SUMMARY")
print(f"{'='*60}")

alloc_counts = {name: [] for name in PRODUCTS}
for result in all_results:
    for name, alloc in result["portfolio"].items():
        alloc_counts[name].append(alloc)

print("\nAverage optimal allocation across all scenarios:")
for name, allocs in sorted(alloc_counts.items(), key=lambda x: -np.mean(x[1])):
    avg = np.mean(allocs)
    std = np.std(allocs)
    direction = "BUY " if np.mean([product_directions[name]]) == 1 else "SELL"
    if avg > 0:
        print(f"  {name:<20} avg: {avg:>5.1f}%  std: {std:>4.1f}%")

print("\nMean PnL per scenario:")
for result in all_results:
    print(f"  {result['scenario']:<35} mean PnL: {result['mean_pnl']:>10,.0f}  fees: {result['total_fees']:>8,.0f}")


Scenario: rat2000_norm1000_id500

Expected returns per product:
  Thermalite           mean return:   0.414  direction: BUY
  LavaCakes            mean return:  -0.426  direction: SELL
  Pyroflex             mean return:  -0.229  direction: SELL
  Sulfur               mean return:   0.143  direction: BUY
  LavaFountainPen      mean return:   0.169  direction: BUY
  VolcanicIncense      mean return:   0.010  direction: BUY
  AshesOfPhoenix       mean return:  -0.091  direction: SELL
  ObsidianCutlery      mean return:  -0.067  direction: SELL
  ScoriaPaste          mean return:   0.057  direction: BUY
Stage 1: coarse sweep high conviction...
Stage 1 best mean: 105,485
Stage 1 allocations: {'Thermalite': 20, 'LavaCakes': 20, 'Pyroflex': 10, 'Sulfur': 5}

Stage 2: sweep low conviction with remaining budget...

Stage 3: fine tuning at 1% steps...

Stage 3: Fine-tuning (Finding Optimal Total Budget)...

── best portfolio ───────────────────────────────────────
  Thermalite           BUY   

In [ ]:
rng = np.random.default_rng(42)

BUDGET = 1_000_000
N_TRIALS = 50_000  # reduced for speed given portfolio sweep

PRODUCTS = {
    "Thermalite":      {"min_r": -0.10, "max_r": 0.60, "anchor": 0.15,
                        "rational": 0.9,  "normal": 0.7,  "idiot_type": "random",
                        "awareness": {"rational": 0.90, "normal": 0.70, "idiot": 0.20}},
    "LavaCakes":       {"min_r": -0.60, "max_r": 0.10, "anchor": -0.15,
                        "rational": -0.9, "normal": -0.8, "idiot_type": "random",
                        "awareness": {"rational": 0.90, "normal": 0.75, "idiot": 0.15}},
    "Pyroflex":        {"min_r": -0.50, "max_r": 0.05, "anchor": -0.10,
                        "rational": -0.8, "normal": -0.6, "idiot_type": "random",
                        "awareness": {"rational": 0.60, "normal": 0.30, "idiot": 0.05}},
    "Sulfur":          {"min_r": -0.05, "max_r": 0.30, "anchor": 0.10,
                        "rational": 0.7,  "normal": 0.3,  "idiot_type": "random",
                        "awareness": {"rational": 0.50, "normal": 0.20, "idiot": 0.05}},
    "LavaFountainPen": {"min_r": -0.10, "max_r": 0.40, "anchor": 0.05,
                        "rational": 0.4,  "normal": 0.6,  "idiot_type": "hype",
                        "awareness": {"rational": 0.60, "normal": 0.70, "idiot": 0.80}},
    "VolcanicIncense": {"min_r": -0.30, "max_r": 0.40, "anchor": 0.00,
                        "rational": -0.5, "normal": 0.3,  "idiot_type": "nostralico",
                        "awareness": {"rational": 0.30, "normal": 0.15, "idiot": 0.85}},
    "AshesOfPhoenix":  {"min_r": -0.30, "max_r": 0.15, "anchor": -0.05,
                        "rational": -0.5, "normal": -0.2, "idiot_type": "random",
                        "awareness": {"rational": 0.50, "normal": 0.40, "idiot": 0.20}},
    "ObsidianCutlery": {"min_r": -0.25, "max_r": 0.10, "anchor": -0.05,
                        "rational": -0.5, "normal": 0.0,  "idiot_type": "random",
                        "awareness": {"rational": 0.30, "normal": 0.10, "idiot": 0.05}},
    "ScoriaPaste":     {"min_r": -0.10, "max_r": 0.25, "anchor": 0.05,
                        "rational": 0.3,  "normal": 0.0,  "idiot_type": "random",
                        "awareness": {"rational": 0.20, "normal": 0.10, "idiot": 0.05}},
}

SEGMENT_SCENARIOS = [
    (2000, 1000, 500),
    (1500, 1000, 1000),
    (2500, 700,  300),
    (1800, 1200, 500),
    (1200, 1300, 1000),  # pessimistic — more idiots and normal
]

def fee(vol_pct):
    return (vol_pct / 100) ** 2 * BUDGET

def simulate_crowd_with_awareness(prod_cfg, n_rational, n_normal, n_idiot, n_trials):
    awareness = prod_cfg["awareness"]

    # rational
    aware_r = rng.random((n_trials, n_rational)) < awareness["rational"]
    r_signal = rng.normal(prod_cfg["rational"], 0.15, (n_trials, n_rational))
    r_random = rng.uniform(-1, 1, (n_trials, n_rational))
    r_votes = np.where(aware_r, r_signal, r_random)
    r_votes = np.clip(r_votes, -1, 1)

    # normal
    aware_n = rng.random((n_trials, n_normal)) < awareness["normal"]
    n_signal = rng.normal(prod_cfg["normal"], 0.30, (n_trials, n_normal))
    n_random = rng.uniform(-1, 1, (n_trials, n_normal))
    n_votes = np.where(aware_n, n_signal, n_random)
    n_votes = np.clip(n_votes, -1, 1)

    # idiots
    aware_i = rng.random((n_trials, n_idiot)) < awareness["idiot"]
    if prod_cfg["idiot_type"] == "hype":
        idiot_signal = rng.uniform(0.5, 1.0, (n_trials, n_idiot))
    elif prod_cfg["idiot_type"] == "nostralico":
        idiot_signal = rng.uniform(0.6, 1.0, (n_trials, n_idiot))
    else:
        idiot_signal = rng.uniform(-1, 1, (n_trials, n_idiot))
    i_random = rng.uniform(-1, 1, (n_trials, n_idiot))
    i_votes = np.where(aware_i, idiot_signal, i_random)
    i_votes = np.clip(i_votes, -1, 1)

    total = n_rational + n_normal + n_idiot
    pressure = (r_votes.sum(axis=1) + n_votes.sum(axis=1) + i_votes.sum(axis=1)) / total
    return np.clip(pressure, -1, 1)

def pressure_to_return(pressure, prod_cfg):
    anchor = prod_cfg["anchor"]
    min_r  = prod_cfg["min_r"]
    max_r  = prod_cfg["max_r"]
    return np.where(
        pressure >= 0,
        anchor + pressure * (max_r - anchor),
        anchor + pressure * (anchor - min_r)
    )

def simulate_returns(n_rational, n_normal, n_idiot, n_trials=N_TRIALS):
    product_returns = {}
    product_directions = {}
    for name, cfg in PRODUCTS.items():
        pressure = simulate_crowd_with_awareness(cfg, n_rational, n_normal, n_idiot, n_trials)
        returns = pressure_to_return(pressure, cfg)
        product_returns[name] = returns
        product_directions[name] = 1 if np.mean(returns) >= 0 else -1
    return product_returns, product_directions

def portfolio_pnl(alloc_dict, product_returns, product_directions):
    pnl = np.zeros(N_TRIALS)
    for name, alloc in alloc_dict.items():
        if alloc == 0:
            continue
        returns = product_returns[name]
        direction = product_directions[name]
        pnl += direction * (alloc / 100) * BUDGET * returns - fee(alloc)
    return pnl


HIGH_CONVICTION  = ["Thermalite", "LavaCakes", "Pyroflex", "Sulfur"]
LOW_CONVICTION   = ["LavaFountainPen", "VolcanicIncense", "AshesOfPhoenix", 
                    "ObsidianCutlery", "ScoriaPaste"]

ALLOC_COARSE = list(range(0, 31, 5))

def efficient_portfolio_sweep(product_returns, product_directions, max_alloc=100):
    products = list(PRODUCTS.keys())
    n = len(products)
    
    best_mean = -np.inf
    best_portfolio = None
    best_pnl = None
    

    print("Stage 1: coarse sweep high conviction...")
    stage1_best = {p: 0 for p in products}
    stage1_best_mean = -np.inf
    
    for allocs in iproduct(ALLOC_COARSE, repeat=len(HIGH_CONVICTION)):
        if sum(allocs) > max_alloc:
            continue
        alloc_dict = {p: 0 for p in products}
        for p, a in zip(HIGH_CONVICTION, allocs):
            alloc_dict[p] = a
        pnl = portfolio_pnl(alloc_dict, product_returns, product_directions)
        m = np.mean(pnl)
        if m > stage1_best_mean:
            stage1_best_mean = m
            stage1_best = alloc_dict.copy()
    
    print(f"Stage 1 best mean: {stage1_best_mean:,.0f}")
    print(f"Stage 1 allocations: { {k:v for k,v in stage1_best.items() if v>0} }")

    print("\nStage 2: sweep low conviction with remaining budget...")
    remaining = max_alloc - sum(stage1_best[p] for p in HIGH_CONVICTION)
    low_alloc_range = list(range(0, min(remaining + 1, 21), 5))
    
    for allocs in iproduct(low_alloc_range, repeat=len(LOW_CONVICTION)):
        if sum(allocs) > remaining:
            continue
        alloc_dict = stage1_best.copy()
        for p, a in zip(LOW_CONVICTION, allocs):
            alloc_dict[p] = a
        pnl = portfolio_pnl(alloc_dict, product_returns, product_directions)
        m = np.mean(pnl)
        if m > best_mean:
            best_mean = m
            best_portfolio = alloc_dict.copy()
            best_pnl = pnl.copy()
    
    return best_portfolio, best_mean, best_pnl

# stage 3 — fine tune around best portfolio at 1% steps
def fine_tune(best_portfolio, product_returns, product_directions, window=4):
    print("\nStage 3: fine tuning at 1% steps...")
    best_mean = np.mean(portfolio_pnl(best_portfolio, product_returns, product_directions))
    improved = True
    
    while improved:
        improved = False
        for name in PRODUCTS:
            current = best_portfolio[name]
            for delta in range(-window, window + 1):
                if delta == 0:
                    continue
                new_alloc = current + delta
                if new_alloc < 0:
                    continue
                test = best_portfolio.copy()
                test[name] = new_alloc
                if sum(test.values()) > 100:
                    continue
                pnl = portfolio_pnl(test, product_returns, product_directions)
                m = np.mean(pnl)
                if m > best_mean:
                    best_mean = m
                    best_portfolio = test.copy()
                    improved = True
    
    return best_portfolio, best_mean

def fine_tune_variable_budget(best_portfolio, product_returns, product_directions):
    print("\nStage 3: Fine-tuning (Finding Optimal Total Budget)...")
    current_portfolio = best_portfolio.copy()

    for name in PRODUCTS:
        if name not in current_portfolio:
            current_portfolio[name] = 0
            
    best_mean = np.mean(portfolio_pnl(current_portfolio, product_returns, product_directions))
    
    improved = True
    while improved:
        improved = False
        for name in PRODUCTS:

            for delta in [-3, -2, -1, 1, 2, 3]:
                test_portfolio = current_portfolio.copy()
                test_portfolio[name] += delta

                if test_portfolio[name] < 0 or sum(test_portfolio.values()) > 100:
                    continue
                
                new_mean = np.mean(portfolio_pnl(test_portfolio, product_returns, product_directions))
                
                if new_mean > best_mean:
                    best_mean = new_mean
                    current_portfolio = test_portfolio.copy()
                    print(f"  Nudge: {name} by {delta:+}% | Total Alloc: {sum(current_portfolio.values())}% | Mean: {best_mean:,.0f}")
                    improved = True
                    break
            if improved: break
            
    return current_portfolio, best_mean

all_results = []

for n_rat, n_norm, n_id in SEGMENT_SCENARIOS:
    scenario_name = f"rat{n_rat}_norm{n_norm}_id{n_id}"
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    print(f"{'='*60}")
    
    product_returns, product_directions = simulate_returns(n_rat, n_norm, n_id)
    
    print(f"\nExpected returns per product:")
    for name, returns in product_returns.items():
        print(f"  {name:<20} mean return: {np.mean(returns):>7.3f}  direction: {'BUY' if product_directions[name]==1 else 'SELL'}")
    
    best_portfolio, best_mean, best_pnl = efficient_portfolio_sweep(
        product_returns, product_directions
    )
    '''
    best_portfolio, best_mean = fine_tune(
        best_portfolio, product_returns, product_directions
    )
    
    # ... after Stage 2 ...
    best_portfolio, best_mean =fine_tune_variable_budget(
        best_portfolio, product_returns, product_directions
    )
        # run fine tune multiple times from different starting points
    '''
    starting_points = [
        best_portfolio,  # from sweep
        {p: 10 for p in PRODUCTS},  # flat start
        {p: 0 for p in PRODUCTS},   # empty start
    ]
    
    results = []
    for start in starting_points:
        portfolio, mean = fine_tune_variable_budget(
            start, product_returns, product_directions
        )
        results.append((portfolio, mean))

    best_portfolio, best_mean = max(results, key=lambda x: x[1])
    
    final_pnl = portfolio_pnl(best_portfolio, product_returns, product_directions)
    
    print(f"\n── best portfolio ───────────────────────────────────────")
    for name, alloc in best_portfolio.items():
        if alloc > 0:
            direction = "BUY" if product_directions[name] == 1 else "SELL"
            f = fee(alloc)
            print(f"  {name:<20} {direction:<5} {alloc:>3}%  fee: {f:>8,.0f}")
    
    total_alloc = sum(best_portfolio.values())
    total_fees  = sum(fee(a) for a in best_portfolio.values())
    
    print(f"\n  Total allocation: {total_alloc}%")
    print(f"  Total fees:       {total_fees:>10,.0f}")
    print(f"  Mean PnL:         {np.mean(final_pnl):>10,.0f}")
    print(f"  Std PnL:          {np.std(final_pnl):>10,.0f}")
    print(f"  P10:              {np.percentile(final_pnl, 10):>10,.0f}")
    print(f"  P90:              {np.percentile(final_pnl, 90):>10,.0f}")
    print(f"  Sharpe:           {np.mean(final_pnl)/np.std(final_pnl):>10.4f}")
    
    all_results.append({
        "scenario": scenario_name,
        "portfolio": best_portfolio,
        "mean_pnl": round(np.mean(final_pnl), 0),
        "std_pnl": round(np.std(final_pnl), 0),
        "total_alloc": total_alloc,
        "total_fees": round(total_fees, 0),
    })

print(f"\n{'='*60}")
print("CROSS SCENARIO SUMMARY")
print(f"{'='*60}")

# average allocation across scenarios
alloc_counts = {name: [] for name in PRODUCTS}
for result in all_results:
    for name, alloc in result["portfolio"].items():
        alloc_counts[name].append(alloc)

print("\nAverage optimal allocation across all scenarios:")
for name, allocs in sorted(alloc_counts.items(), key=lambda x: -np.mean(x[1])):
    avg = np.mean(allocs)
    std = np.std(allocs)
    direction = "BUY " if np.mean([product_directions[name]]) == 1 else "SELL"
    if avg > 0:
        print(f"  {name:<20} avg: {avg:>5.1f}%  std: {std:>4.1f}%")

print("\nMean PnL per scenario:")
for result in all_results:
    print(f"  {result['scenario']:<35} mean PnL: {result['mean_pnl']:>10,.0f}  fees: {result['total_fees']:>8,.0f}")


Scenario: rat2000_norm1000_id500

Expected returns per product:
  Thermalite           mean return:   0.414  direction: BUY
  LavaCakes            mean return:  -0.426  direction: SELL
  Pyroflex             mean return:  -0.229  direction: SELL
  Sulfur               mean return:   0.143  direction: BUY
  LavaFountainPen      mean return:   0.169  direction: BUY
  VolcanicIncense      mean return:   0.010  direction: BUY
  AshesOfPhoenix       mean return:  -0.091  direction: SELL
  ObsidianCutlery      mean return:  -0.067  direction: SELL
  ScoriaPaste          mean return:   0.057  direction: BUY
Stage 1: coarse sweep high conviction...
Stage 1 best mean: 105,485
Stage 1 allocations: {'Thermalite': 20, 'LavaCakes': 20, 'Pyroflex': 10, 'Sulfur': 5}

Stage 2: sweep low conviction with remaining budget...

Stage 3: Fine-tuning (Finding Optimal Total Budget)...
  Nudge: Thermalite by +1% | Total Alloc: 81% | Mean: 115,704
  Nudge: LavaCakes by +1% | Total Alloc: 82% | Mean: 115,862
  

In [14]:
x = pd.DataFrame(all_results)
x

,scenario,portfolio,mean_pnl,std_pnl,total_alloc,total_fees
0,rat2000_norm1000_id500,"{'Thermalite': 21, 'LavaCakes': 21, 'Pyroflex'...",117457.0,1084.0,79,115900.0
1,rat1500_norm1000_id1000,"{'Thermalite': 18, 'LavaCakes': 19, 'Pyroflex'...",95717.0,1039.0,76,95800.0
2,rat2500_norm700_id300,"{'Thermalite': 22, 'LavaCakes': 23, 'Pyroflex'...",133588.0,1122.0,87,136100.0
3,rat1800_norm1200_id500,"{'Thermalite': 20, 'LavaCakes': 21, 'Pyroflex'...",113764.0,1083.0,79,112700.0
4,rat1200_norm1300_id1000,"{'Thermalite': 18, 'LavaCakes': 18, 'Pyroflex'...",90981.0,1045.0,75,92700.0


In [15]:
best_portfolio

{'Thermalite': 18,
 'LavaCakes': 18,
 'Pyroflex': 10,
 'Sulfur': 6,
 'LavaFountainPen': 10,
 'VolcanicIncense': 3,
 'AshesOfPhoenix': 4,
 'ObsidianCutlery': 3,
 'ScoriaPaste': 3}

In [17]:
x = pd.DataFrame(results)
x

,0,1
0,"{'Thermalite': 18, 'LavaCakes': 18, 'Pyroflex'...",90980.924161
1,"{'Thermalite': 18, 'LavaCakes': 18, 'Pyroflex'...",90980.924161
2,"{'Thermalite': 18, 'LavaCakes': 18, 'Pyroflex'...",90980.924161
